# Laboratorio 5 - Reto Babel
## Aplicación de Atención y Transformers

**Modalidad:** grupos de hasta 3 integrantes  
**Entrega:** domingo 6 de septiembre de 2026, 23:59  
**Archivo:** `S09_Lab05_Reto_Babel_ESTUDIANTE.ipynb`  
**Valor:** 100 puntos

Su misión es entrenar un traductor de un idioma inventado al español utilizando PyTorch.

El diccionario ayuda a reconocer las palabras, pero el modelo debe aprender a reorganizarlas. En el
idioma secreto el verbo aparece al final, el adjetivo antes del sustantivo y la negación después del verbo.

### Reglas

- Use únicamente PyTorch y la biblioteca estándar de Python. No use NumPy.
- Use `nn.Transformer`; no reimplemente internamente la atención multi-cabeza.
- No use modelos preentrenados, traductores externos ni APIs generativas.
- No modifique las celdas de verificación.
- Entregue el notebook ejecutado, con las respuestas y salidas visibles.

## Bloque 0 - Investigación guiada (20 puntos)

Lea las secciones 1, 3.1, 3.2 y 3.5 de [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762)
y consulte la [documentación de `nn.Transformer`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Transformer.html).

Responda con lenguaje sencillo y cite el enlace utilizado. Estas son las **únicas cuatro preguntas de
investigación** del laboratorio:

1. ¿Qué limitación de los modelos recurrentes buscaba resolver el Transformer y por qué la traducción era una tarea importante para demostrarlo?
2. ¿Qué trabajo realizan el encoder, la self-attention enmascarada del decoder y la atención entre decoder y encoder en un traductor?
3. ¿Por qué el Transformer necesita información posicional y qué espera que ocurra en este laboratorio si se elimina?
4. ¿Qué ventaja y qué costo de la atención frente a la recurrencia identifica en el artículo, y cómo se relacionan con las frases cortas del Reto Babel?

### Respuestas del grupo

**1.** El Transformer buscaba quitar la dependencia secuencial de los modelos recurrentes. En una RNN el estado de la posición t necesita el estado de t-1, así que las palabras de una frase se procesan una tras otra y no se pueden repartir en paralelo dentro del mismo ejemplo. Eso limita el tamaño de los lotes con frases largas y alarga el entrenamiento. Además, la información entre dos palabras lejanas tiene que atravesar muchos pasos intermedios y se va perdiendo. La atención conecta cualquier par de posiciones en un solo paso, y todas las posiciones se calculan a la vez. La traducción era la tarea para demostrarlo porque tenía puntos de comparación claros (WMT 2014 inglés-alemán e inglés-francés), porque obliga a reordenar la frase y no solo a copiarla, y porque hasta ese momento los mejores resultados venían justamente de arquitecturas recurrentes con atención. Superarlas sin recurrencia era la prueba más directa de que la atención bastaba.

**Fuente:** https://arxiv.org/abs/1706.03762 (sección 1)

**2.** El encoder lee la frase de origen completa y produce una representación por token, donde cada token ya incorpora contexto de los demás. Como no hay máscara causal, la posición 1 puede mirar la posición 7 y al revés, que es lo que aquí permite detectar el verbo al final de la frase secreta. La self-attention enmascarada del decoder hace lo mismo sobre la frase española que se está generando, pero con una máscara que tapa las posiciones futuras. Sin esa máscara el modelo vería la palabra que debe predecir y el entrenamiento no serviría, porque en inferencia esas palabras todavía no existen. La atención entre decoder y encoder es la que une las dos frases: las consultas salen del decoder y las claves y valores salen de la salida del encoder, así que al escribir cada palabra española el modelo decide de qué partes de la frase secreta depende. Esa capa es la que hace el reordenamiento sujeto-objeto-verbo a sujeto-verbo-objeto, porque nada la obliga a leer el origen en orden.

**Fuente:** https://arxiv.org/abs/1706.03762 (secciones 3.1 y 3.2) y https://docs.pytorch.org/docs/stable/generated/torch.nn.Transformer.html

**3.** Porque la atención no distingue el orden por sí sola. La atención calcula un promedio ponderado sobre todas las posiciones, y esa operación es la misma si se barajan las entradas, así que sin información posicional el modelo ve la frase como un conjunto de palabras y no como una secuencia. La codificación posicional sinusoidal suma a cada embedding un vector que depende del índice, y eso rompe el empate. En este laboratorio la posición es especialmente importante porque las tres reglas del idioma secreto son reglas de orden y no de vocabulario: el verbo va al final, el adjetivo va antes del sustantivo y la negación va después del verbo. El diccionario ya resuelve qué significa cada palabra, lo único que falta es dónde ponerla. Esperamos que al desactivar la posición el modelo siga acertando las palabras sueltas, porque puede aprender la correspondencia token a token, pero que se equivoque al colocarlas, sobre todo en el verbo y en la negación. En números esperamos que la exactitud por token baje poco y que el porcentaje de frases completamente correctas se caiga mucho, porque una frase solo cuenta si todas sus posiciones están bien.

**Fuente:** https://arxiv.org/abs/1706.03762 (sección 3.5)

**4.** La ventaja es el costo por capa y el camino corto entre posiciones. La tabla 1 del artículo compara self-attention con recurrencia: la self-attention necesita O(1) operaciones secuenciales frente a O(n) de una RNN, y la distancia máxima entre dos posiciones cualesquiera es O(1) frente a O(n). El costo es que la complejidad por capa pasa a ser O(n²·d), porque cada posición se compara contra todas las demás, mientras que la recurrencia es O(n·d²). Es decir, la atención es más barata cuando la longitud n es menor que la dimensión d, y se vuelve cara cuando las frases son largas. En el Reto Babel esto juega totalmente a favor: las frases tienen unos 6 a 8 tokens y d_modelo es 48, así que n es mucho menor que d y el término cuadrático no se nota. Con esas longitudes el modelo entero cabe en menos de 250 000 parámetros y entrena en segundos, mientras que el camino de un token a otro es siempre un solo salto, justo lo que hace falta para mover el verbo del final de la frase al centro.

**Fuente:** https://arxiv.org/abs/1706.03762 (sección 1 y tabla 1)

## Bloque 1 - Preparar el entorno y explorar los datos (5 puntos)

Cargue los tres CSV recibidos. Verifique cuántos ejemplos tiene y observe cómo cambia el orden entre
el idioma secreto y el español. El diccionario es una ayuda; el modelo debe aprender la transformación.

In [ ]:
from __future__ import annotations

import csv
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

SEMILLA = 42
random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DISPOSITIVO)

# Ejemplos de funciones utilizadas en este bloque:
# torch.manual_seed(...), torch.device(...), torch.cuda.is_available()

In [ ]:
CARPETA_DATOS = Path("Datos")

# Ejemplos de funciones utilizadas en este bloque:
# Path("carpeta"), Path.exists(), Path / "archivo.csv"
assert CARPETA_DATOS.exists(), "Actualice CARPETA_DATOS con la carpeta recibida."

In [ ]:
def leer_pares(ruta: Path):
    with ruta.open(encoding="utf-8", newline="") as archivo:
        return list(csv.DictReader(archivo))


class Vocabulario:
    ESPECIALES = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

    def __init__(self, tokens):
        unicos = self.ESPECIALES + sorted(set(tokens) - set(self.ESPECIALES))
        self.token_a_id = {token: i for i, token in enumerate(unicos)}
        self.id_a_token = {i: token for token, i in self.token_a_id.items()}

    def __len__(self):
        return len(self.token_a_id)

    def codificar(self, texto, agregar_inicio_fin=True):
        ids = [self.token_a_id.get(token, self.token_a_id["<UNK>"]) for token in texto.split()]
        if agregar_inicio_fin:
            ids = [self.token_a_id["<SOS>"]] + ids + [self.token_a_id["<EOS>"]]
        return ids

    def decodificar(self, ids):
        tokens = []
        for indice in ids:
            token = self.id_a_token[int(indice)]
            if token == "<EOS>":
                break
            if token not in {"<PAD>", "<SOS>"}:
                tokens.append(token)
        return " ".join(tokens)


class ParesTraduccion(Dataset):
    def __init__(self, pares, vocab_src, vocab_tgt):
        self.pares = pares
        self.vocab_src = vocab_src
        self.vocab_tgt = vocab_tgt

    def __len__(self):
        return len(self.pares)

    def __getitem__(self, indice):
        fila = self.pares[indice]
        src = torch.tensor(self.vocab_src.codificar(fila["idioma_secreto"]), dtype=torch.long)
        tgt = torch.tensor(self.vocab_tgt.codificar(fila["espanol"]), dtype=torch.long)
        return src, tgt

In [ ]:
pares_entrenamiento = leer_pares(CARPETA_DATOS / "entrenamiento.csv")
pares_validacion = leer_pares(CARPETA_DATOS / "validacion.csv")
diccionario = leer_pares(CARPETA_DATOS / "diccionario.csv")

tokens_src = [token for fila in pares_entrenamiento for token in fila["idioma_secreto"].split()]
tokens_tgt = [token for fila in pares_entrenamiento for token in fila["espanol"].split()]
vocab_src = Vocabulario(tokens_src)
vocab_tgt = Vocabulario(tokens_tgt)

print("Diccionario:", len(diccionario), "palabras")
print("Entrenamiento:", len(pares_entrenamiento), "frases")
print("Validación:", len(pares_validacion), "frases")
print("Ejemplo:", pares_entrenamiento[0])

assert len(diccionario) >= 20
assert len(pares_entrenamiento) >= 1000
assert {"<PAD>", "<SOS>", "<EOS>"}.issubset(vocab_tgt.token_a_id)
print("OK - datos y vocabularios cargados")

# Ejemplos de funciones utilizadas en este bloque:
# leer_pares(...), str.split(), Vocabulario(...), len(...)

## Bloque 2 - Lotes, padding y máscaras (15 puntos)

Prepare lotes de largo variable. La máscara de padding evita que el Transformer trate `<PAD>` como
una palabra real. La máscara causal evita que el decoder observe palabras futuras.

In [ ]:
def crear_collate(pad_src, pad_tgt):
    def collate(lote):
        srcs, tgts = zip(*lote)
        src_lote = pad_sequence(srcs, batch_first=True, padding_value=pad_src)
        tgt_lote = pad_sequence(tgts, batch_first=True, padding_value=pad_tgt)
        return src_lote, tgt_lote
    return collate


def crear_mascara_causal(tamano, device):
    unos = torch.ones(tamano, tamano, dtype=torch.bool, device=device)
    return torch.triu(unos, diagonal=1)


def crear_mascara_padding(tokens, pad_idx):
    return tokens.eq(pad_idx)

In [ ]:
PAD_SRC = vocab_src.token_a_id["<PAD>"]
PAD_TGT = vocab_tgt.token_a_id["<PAD>"]
collate_fn = crear_collate(PAD_SRC, PAD_TGT)

ds_entrenamiento = ParesTraduccion(pares_entrenamiento, vocab_src, vocab_tgt)
ds_validacion = ParesTraduccion(pares_validacion, vocab_src, vocab_tgt)
dl_entrenamiento = DataLoader(ds_entrenamiento, batch_size=64, shuffle=True, collate_fn=collate_fn)
dl_validacion = DataLoader(ds_validacion, batch_size=128, shuffle=False, collate_fn=collate_fn)

src_prueba, tgt_prueba = next(iter(dl_entrenamiento))
mascara = crear_mascara_causal(tgt_prueba.size(1) - 1, torch.device("cpu"))

assert src_prueba.ndim == 2 and tgt_prueba.ndim == 2
assert mascara.shape == (tgt_prueba.size(1) - 1, tgt_prueba.size(1) - 1)
assert mascara.dtype == torch.bool
assert not mascara.diag().any() and mascara[0, -1]
assert crear_mascara_padding(src_prueba, PAD_SRC).shape == src_prueba.shape
print("OK - lotes y máscaras correctos", src_prueba.shape, tgt_prueba.shape)

# Ejemplos de funciones utilizadas en este bloque:
# DataLoader(...), next(iter(...)), Tensor.size(...), torch.device(...)

## Bloque 3 - Configurar el traductor con `nn.Transformer` (25 puntos)

No construya la atención internamente. Ensamble embeddings, posición, `nn.Transformer` y la capa que
predice la siguiente palabra española.

In [ ]:
class CodificacionPosicional(nn.Module):
    """Codificación sinusoidal del artículo original, ya implementada."""

    def __init__(self, d_modelo, max_largo=64, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        posicion = torch.arange(max_largo, dtype=torch.float32).unsqueeze(1)
        divisor = torch.exp(
            torch.arange(0, d_modelo, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_modelo)
        )
        pe = torch.zeros(1, max_largo, d_modelo)
        pe[0, :, 0::2] = torch.sin(posicion * divisor)
        pe[0, :, 1::2] = torch.cos(posicion * divisor)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

In [ ]:
class TraductorTransformer(nn.Module):
    def __init__(self, n_src, n_tgt, pad_src, pad_tgt, d_modelo=48, cabezas=4,
                 capas=2, d_ff=96, dropout=0.1, usar_posicion=True):
        super().__init__()
        self.d_modelo = d_modelo
        self.pad_src = pad_src
        self.pad_tgt = pad_tgt
        self.usar_posicion = usar_posicion

        self.emb_src = nn.Embedding(n_src, d_modelo, padding_idx=pad_src)
        self.emb_tgt = nn.Embedding(n_tgt, d_modelo, padding_idx=pad_tgt)
        self.posicion = CodificacionPosicional(d_modelo, dropout=dropout)
        self.transformer = nn.Transformer(
            d_model=d_modelo,
            nhead=cabezas,
            num_encoder_layers=capas,
            num_decoder_layers=capas,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.salida = nn.Linear(d_modelo, n_tgt)

    def forward(self, src, tgt_entrada):
        escala = math.sqrt(self.d_modelo)
        x_src = self.emb_src(src) * escala
        x_tgt = self.emb_tgt(tgt_entrada) * escala
        if self.usar_posicion:
            x_src = self.posicion(x_src)
            x_tgt = self.posicion(x_tgt)
        else:
            # Se aplica el mismo dropout sin sumar posicion para que la ablacion sea justa
            x_src = self.posicion.dropout(x_src)
            x_tgt = self.posicion.dropout(x_tgt)

        mascara_causal = crear_mascara_causal(tgt_entrada.size(1), tgt_entrada.device)
        pad_src = crear_mascara_padding(src, self.pad_src)
        pad_tgt = crear_mascara_padding(tgt_entrada, self.pad_tgt)

        estados = self.transformer(
            x_src,
            x_tgt,
            tgt_mask=mascara_causal,
            src_key_padding_mask=pad_src,
            tgt_key_padding_mask=pad_tgt,
            memory_key_padding_mask=pad_src,
        )
        return self.salida(estados)

In [ ]:
modelo_prueba = TraductorTransformer(
    len(vocab_src), len(vocab_tgt), PAD_SRC, PAD_TGT, dropout=0.0
)
logits_prueba = modelo_prueba(src_prueba[:3], tgt_prueba[:3, :-1])

assert logits_prueba.shape == (3, tgt_prueba.size(1) - 1, len(vocab_tgt))
assert sum(p.numel() for p in modelo_prueba.parameters()) < 250_000
print("OK - modelo conectado; parámetros:", sum(p.numel() for p in modelo_prueba.parameters()))

# Ejemplos de funciones utilizadas en este bloque:
# TraductorTransformer(...), Tensor.numel(), model.parameters()

## Bloque 4 - Entrenamiento (15 puntos)

Entrene con *teacher forcing*: el decoder recibe la oración española sin el último token y predice
la misma oración desplazada, sin el primer token.

In [ ]:
def ejecutar_epoca(modelo, loader, criterio, optimizador=None):
    entrenando = optimizador is not None
    modelo.train(entrenando)
    total_perdida = 0.0
    total_tokens = 0

    for src, tgt in loader:
        src, tgt = src.to(DISPOSITIVO), tgt.to(DISPOSITIVO)
        tgt_entrada = tgt[:, :-1]
        tgt_esperado = tgt[:, 1:]

        with torch.set_grad_enabled(entrenando):
            logits = modelo(src, tgt_entrada)
            perdida = criterio(
                logits.reshape(-1, logits.size(-1)),
                tgt_esperado.reshape(-1),
            )

        if entrenando:
            optimizador.zero_grad()
            perdida.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            optimizador.step()

        tokens = tgt_esperado.ne(criterio.ignore_index).sum().item()
        total_perdida += perdida.item() * tokens
        total_tokens += tokens

    return total_perdida / total_tokens

In [ ]:
def entrenar_modelo(usar_posicion=True, epocas=22):
    torch.manual_seed(SEMILLA)
    modelo = TraductorTransformer(
        len(vocab_src), len(vocab_tgt), PAD_SRC, PAD_TGT,
        d_modelo=48, cabezas=4, capas=2, d_ff=96,
        dropout=0.1, usar_posicion=usar_posicion,
    ).to(DISPOSITIVO)
    criterio = nn.CrossEntropyLoss(ignore_index=PAD_TGT)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=2e-3)
    historia = {"train": [], "val": []}
    mejor_estado = None
    mejor_val = float("inf")

    for epoca in range(1, epocas + 1):
        perdida_train = ejecutar_epoca(modelo, dl_entrenamiento, criterio, optimizador)
        perdida_val = ejecutar_epoca(modelo, dl_validacion, criterio)
        historia["train"].append(perdida_train)
        historia["val"].append(perdida_val)
        if perdida_val < mejor_val:
            mejor_val = perdida_val
            mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
        if epoca == 1 or epoca % 5 == 0 or epoca == epocas:
            print(f"época {epoca:02d} | train {perdida_train:.4f} | val {perdida_val:.4f}")

    modelo.load_state_dict(mejor_estado)
    return modelo, historia


modelo, historia = entrenar_modelo(usar_posicion=True)
assert len(historia["train"]) == 22
assert historia["train"][-1] < historia["train"][0]
print("OK - entrenamiento completado")

plt.plot(historia["train"], label="entrenamiento")
plt.plot(historia["val"], label="validación")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.show()

# Ejemplos de funciones utilizadas en este bloque:
# nn.CrossEntropyLoss(...), torch.optim.Adam(...), model.state_dict()
# Tensor.detach().cpu().clone(), plt.plot(...)

## Bloque 5 - Traducir palabra por palabra (10 puntos)

Empiece el decoder con `<SOS>`. En cada paso agregue la palabra con mayor logit y deténgase al producir
`<EOS>` o alcanzar el largo máximo.

In [ ]:
@torch.no_grad()
def traducir(modelo, frase_secreta, max_largo=16):
    modelo.eval()
    src = torch.tensor(
        [vocab_src.codificar(frase_secreta)], dtype=torch.long, device=DISPOSITIVO
    )
    generados = [vocab_tgt.token_a_id["<SOS>"]]

    for _ in range(max_largo):
        entrada = torch.tensor([generados], dtype=torch.long, device=DISPOSITIVO)
        logits = modelo(src, entrada)
        siguiente = logits[0, -1].argmax().item()
        generados.append(siguiente)
        if siguiente == vocab_tgt.token_a_id["<EOS>"]:
            break

    return vocab_tgt.decodificar(generados[1:])

In [ ]:
def evaluar_traducciones(modelo, pares, limite=None):
    seleccion = pares if limite is None else pares[:limite]
    tokens_correctos = 0
    tokens_totales = 0
    frases_exactas = 0
    for fila in seleccion:
        pred = traducir(modelo, fila["idioma_secreto"])
        real = fila["espanol"]
        frases_exactas += int(pred == real)
        pred_tokens, real_tokens = pred.split(), real.split()
        largo = max(len(pred_tokens), len(real_tokens))
        tokens_correctos += sum(
            i < len(pred_tokens) and i < len(real_tokens) and pred_tokens[i] == real_tokens[i]
            for i in range(largo)
        )
        tokens_totales += largo
    return {
        "exactitud_tokens": tokens_correctos / tokens_totales,
        "frases_exactas": frases_exactas / len(seleccion),
    }


metricas = evaluar_traducciones(modelo, pares_validacion)
print(metricas)
for fila in pares_validacion[:5]:
    print("secreto :", fila["idioma_secreto"])
    print("esperado:", fila["espanol"])
    print("modelo  :", traducir(modelo, fila["idioma_secreto"]), "\n")

assert 0.0 <= metricas["exactitud_tokens"] <= 1.0
assert metricas["exactitud_tokens"] >= 0.70, "El traductor todavía necesita entrenamiento o correcciones."
print("OK - traductor funcional")

# Ejemplos de funciones utilizadas en este bloque:
# traducir(...), str.split(), max(...), sum(...)

## Bloque 6 - Reto de investigación aplicado (5 puntos)

Entrene una segunda configuración con `usar_posicion=False`. Registre la pérdida de validación, la
exactitud por token y el porcentaje de frases exactas de ambos modelos. Escriba un párrafo que compare
el resultado con la predicción presentada en la respuesta 3 del Bloque 0.

La comparación debe usar los mismos datos, semilla, arquitectura y cantidad de épocas. No se califica
que la diferencia tenga una dirección específica; se califica que la comparación sea justa y que la
conclusión coincida con la evidencia.

In [ ]:
modelo_sin_posicion, historia_sin_posicion = entrenar_modelo(usar_posicion=False)
metricas_sin_posicion = evaluar_traducciones(modelo_sin_posicion, pares_validacion)

resultados = {
    "con_posicion": metricas,
    "sin_posicion": metricas_sin_posicion,
}
print(resultados)

# Ejemplos de funciones utilizadas en este bloque:
# entrenar_modelo(usar_posicion=False), evaluar_traducciones(...), print(...)

### Comparación y conclusión

Escriba aquí un párrafo breve que describa los resultados, los conecte con su predicción y señale al
menos una limitación del experimento.

## Bloque 7 - Entrega y concurso (5 puntos)

Pegue las frases secretas proporcionadas por el profesor en la lista siguiente. Ejecute la celda para
producir las traducciones que se usarán en la prueba oculta y en el concurso.

In [ ]:
frases_del_reto = [
    # "pegue aquí la primera frase secreta",
    # "pegue aquí la segunda frase secreta",
    # "pegue aquí la tercera frase secreta",
]

traducciones_del_reto = [traducir(modelo, frase) for frase in frases_del_reto]
for frase, traduccion in zip(frases_del_reto, traducciones_del_reto):
    print(frase, "->", traduccion)

# Ejemplos de funciones utilizadas en este bloque:
# traducir(...), zip(...), print(...)

### Registro de contribuciones

| Integrante | Investigación y aportes concretos |
|---|---|
| Nombre 1 | Complete aquí |
| Nombre 2 | Complete aquí |
| Nombre 3, si aplica | Complete aquí |

Antes de entregar: reinicie el kernel, ejecute todo, confirme los mensajes `OK`, guarde las salidas y
conserve el nombre solicitado. Cada integrante debe poder explicar cualquier bloque del notebook.